In [1]:
# !pip install pandas kaggle trl liger-kernel accelerate
# !kaggle competitions download -c deep-past-initiative-machine-translation
# !unzip deep-past-initiative-machine-translation.zip -d deep_past_data_past_data

import pandas as pd
train_df = pd.read_csv("deep_past_data_past_data/train.csv")
test_df = pd.read_csv("deep_past_data_past_data/test.csv")

In [2]:
train_df

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...


In [3]:
def create_text_column(row):
    if "translation" not in row or pd.isna(row["translation"]):
        return "Akkadian:" + row["transliteration"] + ":" + "English:"
    return "Akkadian:" + row["transliteration"] + ":" + "English: " + row["translation"]

In [4]:
train_df["text"] = train_df.apply(create_text_column, axis=1)
test_df["text"] = test_df.apply(create_text_column, axis=1)

validation_df = train_df.sample(frac=0.1, random_state=0)
train_df = train_df.drop(validation_df.index)

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)

/home/iato/code/Akkadian/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import torch

def train_worker():
    # Safe in spawned processes; will crash if the launcher uses fork.
    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    training_args = SFTConfig(
        output_dir="akkadian-translation-model",
        do_train=True,
        do_eval=True,
        bf16=bf16_ok,
    )

    trainer = SFTTrainer(
        model="LiquidAI/LFM2.5-1.2B-Base",
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        args=training_args,
    )

    trainer.train()
    trainer.save_model(training_args.output_dir)

train_worker()

Truncating eval dataset: 100%|██████████| 156/156 [00:00<00:00, 54690.02 examples/s]


Step,Training Loss
10,2.874000
20,2.096100
30,1.819900
40,1.715100
50,1.607700
60,1.562400
70,1.489000
80,1.440000
90,1.503900
100,1.396200


In [ ]:
from transformers import pipeline, AutoTokenizer

# Load the model and tokenizer from the saved path
model_path = "akkadian-translation-model"

# Load tokenizer explicitly to handle potential regex fixes if needed
tokenizer = AutoTokenizer.from_pretrained(model_path, fix_mistral_regex=True)

# Initialize pipeline on GPU (device=0)
text_generator = pipeline("text-generation", model=model_path, tokenizer=tokenizer, device=0)

# Function to generate translation from the text column
def generate_translation(text):
    # return_full_text=False ensures we only get the generated part (the translation), not the prompt
    result = text_generator(text, max_new_tokens=100, return_full_text=False)
    return result[0]['generated_text'].strip()

# Generate predictions
# We use the 'text' field which contains the prompt format defined in create_text_column
debug_df = train_df.head(10).copy()

def remove_translation_part(text):
    # Remove the "English: " part to only keep the Akkadian transliteration
    if "English:" in text:
        return text.split("English:")[0] + "English:"
    return text

debug_df["text"] = debug_df["text"].apply(remove_translation_part)

debug_df["predicted_translation"] = debug_df["text"].apply(generate_translation)
debug_df["id"] = debug_df.index

# Save result
submission_df = debug_df[["id", "predicted_translation"]]
submission_df.to_csv("akkadian_translations.csv", index=False)

out = debug_df[["transliteration", "translation", "predicted_translation"]]
out.to_csv("akkadian_debug_output.csv", index=False)

Device set to use cuda:0


,transliteration,translation,predicted_translation
0,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...","Seal of Mannum-balum-Aššur, son of Ṣilli-Adad,..."
1,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,Itūr-ilī has received one textile of ordinary ...
2,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...,... he did not give you a textile. He returned...
3,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...","Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
5,um-ma šu-ta-mu-zi e-lá-a en-um-a-šùr ù lá-ma-s...,"From Šu-Tammuzī, Elaya, Ennam-Aššur and Lamass...","From Šu-Tammuzi, Elaya, Ennam-Aššur and Lamass..."
6,KIŠIB a-lá-ḫi-im KIŠIB a-li-li KIŠIB a-bi₄-lá ...,"Seal of Ali-ahum, seal of Ali-ilī, seal of Abi...","Seal of Ali-ahum, seal of Ali-ilī, seal of Abi..."
7,a-na É šál-ma-a-šùr a-na a-wa-tim né-ru-ub-ma ...,We entered Šalim-Aššur's house to settle the c...,We entered Šalim-Aššur's house to settle the c...
8,ṭup-pu-um ša 10 ma-na KÙ.BABBAR ša PÚZUR-a-šur...,A tablet about 10 minas of silver of Puzur-Ass...,Tablet concerning 10 minas of silver of Puzur-...
10,2 pí-ri-kà-ni-e šu-ku-tum 2 pí-ri-kà-ni-e en-n...,2 pirikannus: Šukkutum; 2 pirikannus: Enna-Sue...,2 pirikannus: Šukkutum; 2 pirikannus: Enna-Sue...
